# Week 02 — Android UI Fundamentals
## LeafGuard AI: Exploring Layouts and UI Behavior with Python

This notebook models Android UI concepts using pure Python so you can understand hierarchy, XML structure, colours, and lifecycle transitions before touching Java or Kotlin code.

**Learning outcomes:**
- Represent a View hierarchy as a tree structure
- Parse Android-style XML layouts with `xml.etree.ElementTree`
- Compare Material Design colour roles
- Simulate the Activity lifecycle as a state machine
- Connect UI structure to user experience decisions

---


## Cell 1: Represent an Android View Hierarchy as a Tree

A layout is not just a flat list of widgets. It is a nested tree of containers and child views.


In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class ViewNode:
    name: str
    children: List['ViewNode'] = field(default_factory=list)

    def add(self, *nodes):
        self.children.extend(nodes)
        return self

root = ViewNode('ConstraintLayout').add(
    ViewNode('ImageView'),
    ViewNode('TextView'),
    ViewNode('LinearLayout').add(
        ViewNode('Button(capture)'),
        ViewNode('Button(gallery)')
    )
)

def print_tree(node, indent=0):
    print('  ' * indent + f'- {node.name}')
    for child in node.children:
        print_tree(child, indent + 1)

print('=== VIEW TREE ===')
print_tree(root)


## Cell 2: Parse an Android-Style XML Layout

Android layouts are XML documents. Python's `ElementTree` gives us a simple way to inspect tags, attributes, and nesting.


In [ ]:
import xml.etree.ElementTree as ET

xml_layout = """
<LinearLayout xmlns:android="http://schemas.android.com/apk/res/android"
    android:orientation="vertical"
    android:layout_width="match_parent"
    android:layout_height="match_parent">
    <TextView android:id="@+id/title" android:text="LeafGuard AI" />
    <ImageView android:id="@+id/preview" android:contentDescription="Leaf preview" />
    <Button android:id="@+id/captureButton" android:text="Capture Image" />
</LinearLayout>
"""

root = ET.fromstring(xml_layout)
print(f'Root tag: {root.tag}')
print('Children:')
for child in root:
    print(f'  - {child.tag} -> {child.attrib}')


## Cell 3: Material Design Colour Roles

Material Design is more than choosing pretty colours. Each colour has a role such as primary, secondary, surface, and error.


In [ ]:
material_palette = {
    'primary': '#2E7D32',
    'primaryContainer': '#C8E6C9',
    'secondary': '#1565C0',
    'surface': '#FAFAFA',
    'background': '#F1F8E9',
    'error': '#B00020',
}

def hex_to_rgb(value):
    value = value.lstrip('#')
    return tuple(int(value[i:i+2], 16) for i in (0, 2, 4))

print('=== MATERIAL COLOUR PALETTE ===')
for role, hex_code in material_palette.items():
    print(f'{role:<18} {hex_code:<8} rgb={hex_to_rgb(hex_code)}')


## Cell 4: Estimate Contrast with a Simple Brightness Check

Good UI design also considers readability. A quick brightness estimate can hint at whether a text colour may need to be light or dark.


In [ ]:
def perceived_brightness(hex_code):
    r, g, b = hex_to_rgb(hex_code)
    return 0.299 * r + 0.587 * g + 0.114 * b

print('=== BRIGHTNESS ESTIMATES ===')
for role, hex_code in material_palette.items():
    brightness = perceived_brightness(hex_code)
    suggested_text = 'black text' if brightness > 140 else 'white text'
    print(f'{role:<18} brightness={brightness:6.1f} -> suggest {suggested_text}')


## Cell 5: Simulate the Activity Lifecycle as a State Machine

Android Activities move through well-defined states. Thinking in state transitions helps prevent bugs like refreshing UI at the wrong time.


In [ ]:
lifecycle_events = {
    'launch': 'Created',
    'become_visible': 'Started',
    'gain_focus': 'Resumed',
    'open_dialog': 'Paused',
    'background_app': 'Stopped',
    'return_to_app': 'Resumed',
    'close_app': 'Destroyed',
}

current_state = 'Initial'
print('=== ACTIVITY LIFECYCLE SIMULATION ===')
for event, next_state in lifecycle_events.items():
    print(f'{current_state:>10} --{event}--> {next_state}')
    current_state = next_state


## Cell 6: Count Widgets by Type

Once a layout is parsed, simple analysis can answer design questions such as: how many interactive widgets are present?


In [ ]:
from collections import Counter

widget_counter = Counter(child.tag for child in ET.fromstring(xml_layout))
print('=== WIDGET COUNTS ===')
for widget, count in widget_counter.items():
    print(f'{widget:<12} -> {count}')


## Summary and Quick Quiz

**Key takeaways:**
- Android layouts are trees, not flat screens.
- XML layouts can be inspected like any other structured document.
- Colour roles and lifecycle states are core UI design concepts.

**Quick quiz:**
1. Why is a tree a better model than a list for Android views?
2. Which Material colour role would you use for critical errors?
3. What might go wrong if you update UI while an Activity is stopped?
4. How does XML help separate design from application logic?
